In [10]:
%matplotlib tk
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, RadioButtons
from scipy.signal import find_peaks
import os
import re
from pathlib import Path
import json
import time
import hashlib

# Functions

### Finding Data

In [8]:
CACHE_DIR = "drive_index_cache_files"
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)

def get_all_files(root_path, force_update=False):
    """
    Returns a list of all file paths for the given root_path.
    Creates a specific, hashed cache file for this path in the background.
    """
    # 1. Generate a unique cache filename for THIS path
    # We hash the path string to create a safe filename (e.g. "cache_5d41402abc.json")
    path_str = str(root_path)
    path_hash = hashlib.md5(path_str.encode('utf-8')).hexdigest()
    cache_file = os.path.join(CACHE_DIR, f"cache_{path_hash}.json")

    # 2. Check if cache exists and we are not forcing an update
    if os.path.exists(cache_file) and not force_update:
        print(f"Loading file list from local cache: {cache_file}...")
        t0 = time.time()
        try:
            with open(cache_file, 'r') as f:
                file_list = json.load(f)
            print(f"Loaded {len(file_list)} files in {time.time()-t0:.2f}s")
            return file_list
        except Exception as e:
            print(f"Cache corrupted ({e}), rescanning...")

    # 3. If no cache or force_update, scan the drive
    print(f"Scanning drive (this may take a while)... {root_path}")
    t0 = time.time()
    
    root = Path(root_path)
    if not root.exists():
        print(f"Warning: Path not found: {root_path}")
        return []

    # Recursively find all files
    files = [p.as_posix() for p in root.rglob('*') if p.is_file()]
    
    print(f"Scan complete. Found {len(files)} files in {time.time()-t0:.2f}s")

    # 4. Save to the specific cache file
    print(f"Saving cache to {cache_file}...")
    with open(cache_file, 'w') as f:
        json.dump(files, f)
        
    return files

def find_file_cached(root_path, substring, force_refresh=False):
    # Get the list (manages cache automatically based on root_path)
    all_files = get_all_files(root_path, force_update=force_refresh)
    
    # Perform the search in memory
    matches = [f for f in all_files if substring in f.split('/')[-1]]
    
    return matches

### Loading Data

In [3]:
def load_data(name, path, manual_indices=None, sort=True):
    name_extension = name+".hdf5"
    if not path.startswith("Z:/"):
        path = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)"+path
    filepath = os.path.join(path, name_extension)

    data = h5py.File(filepath, 'r')
    # Extract raw data to inspect
    raw_names = data['Data']['Channel names'][:]
    
    # Clean the names: 
    # 1. Select the first element of the tuple (the name)
    # 2. Decode bytes to string
    # 3. Strip whitespace
    channel_names = [x[0].decode('utf-8').strip() for x in raw_names]
    
    print("Found channels:", channel_names)
    # Output: ['V_SD Left C18 C6', 'PG Virtual', 'V_MG C17', ...]
    
    
    # 2. Define Flexible Patterns (Regex)
    # -----------------------------------
    # These patterns define what you are looking for.
    # .  = any character
    # * = zero or more times
    # |  = OR logic
    # ?  = optional character (good for _ vs space)
    patterns = {
        'V_SD':  r'V.?SD',                      # Matches "V_SD", "V SD"
        'V_PG':  r'PG.?Virtual|V.?PG|V.?CG',    # Matches "PG Virtual" OR "V_PG" OR "V_CG"
        'I_ACx': r'I.?SD.*AC.*x',               # Matches "I", (space/_), "SD", then "AC", then "x"
        'I_SD':  r'I.?SD.*?(C18|C17)',          # Matches "I_SD C18", "I SD C18" [potentially also finds AC!]
        'B':     r'.*Magnet.*|B',                 # Matches anything with "Magnet" in it
    }
    
    # 3. Helper Function to Get Index
    # -------------------------------
    def get_index(pattern, name_list):
        for i, name in enumerate(name_list):
            if re.search(pattern, name, re.IGNORECASE):
                return i
        if pattern not in ["I.?SD.*AC.*x", '.*Magnet.*|B']:
            raise ValueError(f"Could not find channel matching pattern: {pattern}")
        else:
            return -1

    # 4. Map Patterns to Indices
    # --------------------------
    indices = {key: get_index(pat, channel_names) for key, pat in patterns.items()}

    print("Mapped Indices:", indices)
    if manual_indices is not None:
        if len(manual_indices) == len(patterns.keys()):
            indices = {key: idx for key, idx in zip(patterns.keys(), manual_indices)}
            print("Overwrote Indices:", indices)
        else:
            print(f"Could not overwrite: manual_indices has length {len(manual_indices)}, needs length {len(patterns.keys())}")
    # Output: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 5}
    
    
    # 5. Extract Data Using Dynamic Indices
    # -------------------------------------
    # Now use the dictionary indices instead of hardcoded numbers
    dataset = data['Data']['Data']
    
    V_SD  = dataset[:, indices['V_SD'], :] * 2
    V_PG  = dataset[:, indices['V_PG'], :]
    if indices['I_ACx'] == -1:
        I_SD = dataset[:, indices['I_SD'], :]
        dx = V_PG[1, 0]-V_PG[0, 0]
        if dx == 0:
            dx = V_PG[0, 1]-V_PG[0, 0]
        print("dx=", dx)
        I_ACx = np.gradient(I_SD, dx, axis=1) * 1e-8
        print("Manually computed I_AC with np.gradient(I_SD,...).")
    else:
        I_ACx = dataset[:, indices['I_ACx'], :] * 1e-8
    if indices['B'] != -1:
        B = dataset[:, indices['B'], :]
        n_B_values = len(np.unique(B))
        print(f"Detected {n_B_values} unique B-field values: {np.unique(B)}")
        # # We need to only take every n_B_values-th slice to get a single B-field
        # V_SD  = V_SD[:, ::n_B_values]
        # V_PG  = V_PG[:, ::n_B_values]
        # I_ACx = I_ACx[:, ::n_B_values]

    # Convert raw data to physical units
    y_all = V_PG[:, :] # 0:101]  # Plunger Gate (V)
    x_all = V_SD[:, :] # 0:101]  # Energy (meV)
    z_all = 1e12 * I_ACx[:, :] # 0:101]  # pA
    # z_all = np.fliplr(z_all)  # to test weak hole

    is_pg_horizontal = abs(V_PG[0, 1] - V_PG[0, 0]) > 1e-9

    if is_pg_horizontal:
        # CASE 1: Standard Orientation (PG is fast axis/columns)
        print("Detected orientation: PG varies along columns.")
        x_centers = V_PG[0, :]
        y_centers = V_SD[:, 0]
        z_final = z_all  # No transpose needed
        
    else:
        # CASE 2: Transposed Orientation (PG is slow axis/rows)
        print("Detected orientation: PG varies along rows. Transposing data...")
        x_centers = V_PG[:, 0]   # Take the column (vertical variation)
        y_centers = V_SD[0, :]   # Take the row (horizontal variation)
        
        # CRITICAL: Transpose Z so the plotting code (pcolormesh) 
        # still sees X as horizontal and Y as vertical.
        z_final = z_all.T

    # ---------------------------------------------------------
    # 7. SORTING (Fixes pcolormesh warning)
    # ---------------------------------------------------------
    if sort:
        # Ensure x_centers is strictly increasing
        if np.any(np.diff(x_centers) <= 0):
            sort_x = np.argsort(x_centers)
            x_centers = x_centers[sort_x]
            z_final = z_final[:, sort_x]  # Reorder columns of data

        # Ensure y_centers is strictly increasing
        if np.any(np.diff(y_centers) <= 0):
            sort_y = np.argsort(y_centers)
            y_centers = y_centers[sort_y]
            z_final = z_final[sort_y, :]  # Reorder rows of data

    return name, x_centers, y_centers, z_final

# Choosing data

In [13]:
search_dir = "Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE04/"
name_substring = "C0062"

# files = list(Path(search_dir).rglob(f"*{name_substring}*.hdf5"))
files = find_file_cached(search_dir, name_substring, force_refresh=False)

if files:
    for file in files:
        print(file)
else:
    print("No files found.")

Scanning drive (this may take a while)... Z:/POBox/Jonas Gerber/05 - Measurements (Data)/EE04/
Scan complete. Found 553 files in 15.08s
Saving cache to drive_index_cache_files\cache_13bd4ff15a240c96cf1cdcb409115a40.json...
No files found.


In [14]:
name, x_centers, y_centers, z_all = load_data(name='C0062 4VBGm 1e Diamond MG', \
                                              path='/EE01/Fritz2025 Cooldown C/2025/07/Data_0716/', sort=False)
# works well

Found channels: ['V_SD Left C18 C6', 'PG Virtual', 'V_MG C17', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 5, 'I_SD': 4, 'B': -1}
Detected orientation: PG varies along columns.


In [ ]:
name, x_centers, y_centers, z_all = load_data('C0193_8VBG_1e diamond', \
                                              '/EE01/Fritz2025 Cooldown C/2025/07/Data_0725/')
z_all = z_all[::-1, :]  # flip vertically to match expected orientation
# works well

Found channels: ['V_SD Left C18 C6', 'PG virtual', 'V_PG Left C24', 'I SD AC C18 AC x', 'I_SD Left C18']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 3, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [16]:
name, x_centers, y_centers, z_all = load_data('C1108 1h high res diamond', \
                                              '/EE01/Fritz2025 Cooldown C/2025/12/Data_1210/')
# works okay

Found channels: ['PG virtual', 'V_SD Left C18 GND', 'V_PG Left C24', 'V_PG Right C13', 'I SD Right C16', 'I SD C18 GND']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': -1, 'I_SD': 5, 'B': -1}
dx= -0.0020000000000000018
Manually computed I_AC with np.gradient(I_SD,...).
Detected orientation: PG varies along rows. Transposing data...


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0373 6VBG 1h diamond', \
                                              '/EE01/Fritz2025/2025/04/Data_0407/')
# works meeh

Found channels: ['V_SD Left C18 C6', 'V_PG Left C24', 'I_SD Left C18', 'I SD Right C16', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 2, 'B': -1}
Detected orientation: PG varies along columns.


In [104]:
name, x_centers, y_centers, z_all = load_data('B0638 6VBG 1h diamond', \
                                              '/EE01/Fritz2025/2025/05/Data_0521/')
# works okay with 2.9, 3, 2.84

Found channels: ['V_PG Left C24', 'V_SD Left C18 C6', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': 3, 'I_SD': 2, 'B': -1}
Detected orientation: PG varies along rows. Transposing data...


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0591 6VBG 1ediamond HQ', \
                                              '/EE01/Fritz2025/2025/05/Data_0514/') #, manual_indices=[0,1,3,3])
# works well

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0451  6VBG 1h diamond lever arm', \
                                              '/EE01/Fritz2025/2025/04/Data_0427/')
# works well

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'I SD C18 AC x', 'I SD C18 AC theta']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [ ]:
name, x_centers, y_centers, z_all = load_data('B0818 m5VBG 1e diamond check', \
                                              '/EE01/Fritz2025/2025/06/Data_0605/')
# difficult

Found channels: ['V_SD Left C18 C6', 'PG-virtual', 'V_PG Left C24', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': 4, 'I_SD': 3, 'B': -1}
Detected orientation: PG varies along columns.


In [100]:
name, x_centers, y_centers, z_all = load_data('B0902 m5VBG 1h Diamonds BField', \
                                              '/EE01/Fritz2025/2025/06/Data_0615/')

Found channels: ['PG-virtual', 'V_SD Left C18 C6', 'Magnet - B', 'V_PG Left C24', 'I_SD Left C18', 'Value I SD C18 ACx', 'Value I SD C18 ACx', 'I SD C18 AC x', 'I SD Right C16']
Mapped Indices: {'V_SD': 1, 'V_PG': 0, 'I_ACx': 5, 'I_SD': 4, 'B': 2}
Detected 13 unique B-field values: [0.    0.025 0.05  0.075 0.1   0.125 0.15  0.175 0.2   0.225 0.25  0.275
 0.3  ]
Detected orientation: PG varies along rows. Transposing data...


In [99]:
name, x_centers, y_centers, z_all = load_data('B149_Efe4 Diamond 1e BField', \
                                              '/EE04/BigMom2024/2024/12/Data_1206/')


Found channels: ['V_SD C8C17', 'V_CG1', 'B', 'I_SD_C17', 'ISD C8', 'I_MG_test']
Mapped Indices: {'V_SD': 0, 'V_PG': 1, 'I_ACx': -1, 'I_SD': 3, 'B': 2}
dx= 0.000500000000000167
Manually computed I_AC with np.gradient(I_SD,...).
Detected 13 unique B-field values: [0.   0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12]
Detected orientation: PG varies along columns.


# Interactive SOC

In [ ]:
# --- 1. CONFIGURATION & INITIALIZATION ---
# Calculate data bounds automatically
d_x_min, d_x_max = np.min(x_centers), np.max(x_centers)
d_y_min, d_y_max = np.min(y_centers), np.max(y_centers)
d_z_max = np.nanmax(z_all)
x_span = d_x_max - d_x_min

# Set initial parameter guesses
init_params = {
    'L_min': d_x_min + 0.05 * x_span,
    'L_max': d_x_min + 0.30 * x_span,
    'R_min': d_x_max - 0.30 * x_span,
    'R_max': d_x_max - 0.05 * x_span,
    'height': 0.10 * d_z_max,
    'dist': 5,
    'prom': 0.05 * d_z_max,
}

# --- 2. FIGURE & PLOT SETUP ---
fig = plt.figure(figsize=(11, 7))
plt.subplots_adjust(bottom=0.40, top=0.85)

# Main Plot Area [left, bottom, width, height]
ax = fig.add_axes([0.15, 0.35, 0.65, 0.55])

# Background Data
z_original = z_all.copy()
plot_z = z_original if z_original.shape[0] == len(y_centers) else z_original.T
pcm = ax.pcolormesh(x_centers, y_centers, plot_z, cmap='binary', shading='auto')

# Colorbar [left, bottom, width, height]
cbar_ax = fig.add_axes([0.87, 0.40, 0.02, 0.45])
cbar = fig.colorbar(pcm, cax=cbar_ax, label=r"$I_\text{AC,x}$ (pA)")
# cbar_ax.tick_params(labelsize=10)

# Interactive Plot Elements
lines = {
    'L_gs': ax.plot([], [], 'c--', alpha=0.6, label='Left GS')[0],
    'L_es': ax.plot([], [], color='orange', linestyle='--', alpha=0.6, label='Left ES')[0],
    'R_gs': ax.plot([], [], 'm--', alpha=0.6, label='Right GS')[0],
    'R_es': ax.plot([], [], color='red', linestyle='--', alpha=0.6, label='Right ES')[0]
}

points = {
    'L_gs': ax.plot([], [], 'o', color='cyan', ms=3, alpha=0.5)[0],
    'L_es': ax.plot([], [], 'o', color='orange', ms=3, alpha=0.5)[0],
    'R_gs': ax.plot([], [], 'o', color='magenta', ms=3, alpha=0.5)[0],
    'R_es': ax.plot([], [], 'o', color='red', ms=3, alpha=0.5)[0]
}

markers = {
    'ref': ax.plot([], [], 'rx', ms=12, mew=2, label='Reference (E=0)')[0],
    'soc': ax.plot([], [], 'gx', ms=12, mew=2, label='SOC Intersect')[0]
}

arrow_ann = ax.annotate('', xy=(0,0), xytext=(0,0), arrowprops=dict(arrowstyle='<->', color='red', lw=1.5))
text_ann = ax.text(0, 0, '', color='red', fontsize=12, fontweight='bold')

spans = {
    'left': ax.axvspan(init_params['L_min'], init_params['L_max'], color='cyan', alpha=0.1),
    'right': ax.axvspan(init_params['R_min'], init_params['R_max'], color='magenta', alpha=0.1)
}

ax.set_xlabel("Plunger Gate Voltage (V)", fontsize=10, labelpad=10)
ax.set_ylabel("Energy (meV)", fontsize=10, labelpad=10)
ax.legend(loc='upper left', fontsize=8, framealpha=0.9)

# --- 3. WIDGET LAYOUT ---
widgets = {}

def add_slider(label, valmin, valmax, valinit, pos, valstep=None):
    ax_s = fig.add_axes(pos)
    return Slider(ax_s, label, valmin, valmax, valinit=valinit, valstep=valstep)

# A. ZOOM CONTROLS (Top)
widgets['x_min'] = add_slider('X Min', d_x_min, d_x_max, d_x_min, [0.10, 0.96, 0.30, 0.02])
widgets['x_max'] = add_slider('X Max', d_x_min, d_x_max, d_x_max, [0.60, 0.96, 0.30, 0.02])
widgets['y_min'] = add_slider('Y Min', d_y_min, d_y_max, d_y_min, [0.10, 0.93, 0.30, 0.02])
widgets['y_max'] = add_slider('Y Max', d_y_min, d_y_max, d_y_max, [0.60, 0.93, 0.30, 0.02])

# B. ANALYSIS CONTROLS (Bottom)
widgets['L_min'] = add_slider('L ROI Min', d_x_min, d_x_max, init_params['L_min'], [0.15, 0.20, 0.30, 0.02])
widgets['L_max'] = add_slider('L ROI Max', d_x_min, d_x_max, init_params['L_max'], [0.15, 0.15, 0.30, 0.02])
widgets['R_min'] = add_slider('R ROI Min', d_x_min, d_x_max, init_params['R_min'], [0.60, 0.20, 0.30, 0.02])
widgets['R_max'] = add_slider('R ROI Max', d_x_min, d_x_max, init_params['R_max'], [0.60, 0.15, 0.30, 0.02])
widgets['height'] = add_slider('Height', -d_z_max, d_z_max, init_params['height'], [0.15, 0.10, 0.20, 0.02])
widgets['dist']   = add_slider('Dist', 1, len(x_centers) // 2, init_params['dist'], [0.45, 0.10, 0.20, 0.02], valstep=1)
widgets['prom']   = add_slider('Prom', 0, d_z_max, init_params['prom'], [0.75, 0.10, 0.15, 0.02])

# Buttons & Radio
ax_btn = fig.add_axes([0.08, 0.02, 0.18, 0.07])
btn_mc = Button(ax_btn, 'Run MC Error\nSimulation', color='lightgray', hovercolor='lightblue')

ax_abs = fig.add_axes([0.30, 0.02, 0.18, 0.07])
btn_abs = Button(ax_abs, 'Toggle Abs(Z)', color='lightgray', hovercolor='lightblue')
use_abs = False  # default state

ax_center = fig.add_axes([0.52, 0.02, 0.18, 0.07])
btn_center = Button(ax_center, 'Toggle Center(Z)', color='lightgray', hovercolor='lightblue')
use_center = False # default State

ax_median = fig.add_axes([0.74, 0.02, 0.18, 0.07])
btn_median = Button(ax_median, 'Toggle Median(Z)', color='lightgray', hovercolor='lightblue')
use_median = False # default State

ax_radio = fig.add_axes([0.02, 0.25, 0.10, 0.10])
radio = RadioButtons(ax_radio, ('Weak e-', 'Weak h+', 'Strong e-', 'Strong h+'), active=0)

# Store fit results for MC
current_fits = {'L': None, 'R': None}

# --- 4. CORE FUNCTIONS ---

# --- HELPER: CONSTRAINED LEAST SQUARES ---
def constrained_fit(Lgs, Les):
    """
    Fits two lines y1 = m*x1 + c1 and y2 = m*x2 + c2
    sharing the SAME slope 'm'.
    Input: Lgs (N,2) and Les (M,2) arrays of [x, y]
    Returns: m, c1, c2, covariance_matrix
    """
    if len(Lgs) < 2 or len(Les) < 2:
        return None

    # Form matrices for solving: A * p = Y
    # p = [m, c1, c2]
    # Equation 1: m*x_gs + c1 + 0*c2 = y_gs
    # Equation 2: m*x_es + 0*c1 + c2 = y_es
    
    x1, y1 = Lgs[:, 0], Lgs[:, 1]
    x2, y2 = Les[:, 0], Les[:, 1]
    
    # Construct Design Matrix A
    # Col 0: x values (shared slope)
    # Col 1: Indicator for GS (1 for GS points, 0 for ES points)
    # Col 2: Indicator for ES (0 for GS points, 1 for ES points)
    
    A1 = np.vstack([x1, np.ones_like(x1), np.zeros_like(x1)]).T
    A2 = np.vstack([x2, np.zeros_like(x2), np.ones_like(x2)]).T
    A = np.vstack([A1, A2])
    Y = np.concatenate([y1, y2])
    
    # Solve (Least Squares)
    # params = [slope, intercept_GS, intercept_ES]
    try:
        # np.linalg.lstsq returns residuals and rank, we want params
        params, residuals, rank, s = np.linalg.lstsq(A, Y, rcond=None)
        
        # Calculate covariance manually for MC error propagation
        # MSE = Sum of Squared Residuals / (N_points - N_params)
        n_points = len(Y)
        n_params = 3
        if n_points > n_params and len(residuals) > 0:
            mse = residuals[0] / (n_points - n_params)
            # Covariance Matrix = MSE * inv(A.T @ A)
            cov = mse * np.linalg.inv(A.T @ A)
        else:
            cov = np.eye(3) * 1e-6 # Fallback if perfect fit or too few points
            
        return params, cov
    except:
        return None
    
# --- Helper to update slider ranges dynamically ---
def adjust_slider_range(slider, new_max):
    """Updates the max range of a slider to new_max.
       Clamps current value if it exceeds new_max."""
    # Temporarily disable events to prevent recursion loops
    old_eventson = slider.eventson
    slider.eventson = False
    
    # Update range
    slider.valmax = new_max
    slider.ax.set_xlim(slider.valmin, new_max)
    
    # Clamp value
    if slider.val > new_max:
        slider.set_val(new_max)
    
    # Restore events
    slider.eventson = old_eventson

def update_all(val):
    # 1. Update View & Spans
    x_lims = (widgets['x_min'].val, widgets['x_max'].val)
    y_lims = (widgets['y_min'].val, widgets['y_max'].val)
    ax.set_xlim(*x_lims); ax.set_ylim(*y_lims)
    
    l_roi = (widgets['L_min'].val, widgets['L_max'].val)
    r_roi = (widgets['R_min'].val, widgets['R_max'].val)
    spans['left'].set_x(l_roi[0]); spans['left'].set_width(l_roi[1]-l_roi[0])
    spans['right'].set_x(r_roi[0]); spans['right'].set_width(r_roi[1]-r_roi[0])

    col_mask = (x_centers >= x_lims[0]) & (x_centers <= x_lims[1])
    row_mask = (y_centers >= y_lims[0]) & (y_centers <= y_lims[1])

    # Check if we actually have data in the view (avoid empty slice errors)
    if np.any(col_mask) and np.any(row_mask):
        # Extract the subset of data visible on screen
        # plot_z is (Rows/Y, Cols/X), so we slice accordingly
        view_data = plot_z[np.ix_(row_mask, col_mask)]
        
        if view_data.size > 0:
            local_min = np.nanmin(view_data)
            local_max = np.nanmax(view_data)
            
            # 1. Update Colorbar
            pcm.set_clim(local_min, local_max)
            
            # 2. Update Slider Ranges
            # Use the larger of the abs(min) or abs(max) to set the bounds
            abs_max = max(abs(local_min), abs(local_max))
            if abs_max < 1e-12: abs_max = 1.0
            
            # Height: Update Min AND Max to handle negative centering
            # We temporarily disable events to avoid loops
            widgets['height'].eventson = False
            widgets['height'].valmin = -abs_max
            widgets['height'].valmax = abs_max
            widgets['height'].ax.set_xlim(-abs_max, abs_max)
            widgets['height'].eventson = True
            
            # Prominence: Scale to the new signal magnitude
            adjust_slider_range(widgets['prom'], abs_max)
            
            # Distance: Scale to the number of visible pixels
            # We use the 'width' of the view (number of columns) as the reference
            n_pixels = view_data.shape[1] 
            new_dist_max = max(1, n_pixels // 2) # Allow at least 2 peaks in view
            adjust_slider_range(widgets['dist'], new_dist_max)
    
    h, d, p = widgets['height'].val, widgets['dist'].val, widgets['prom'].val
    regime = radio.value_selected 

    # 2. Universal Point Finder
    def extract_pts(roi):
        eff_min = max(roi[0], x_lims[0])
        eff_max = min(roi[1], x_lims[1])
        col_mask = (x_centers >= eff_min) & (x_centers <= eff_max)
        row_mask = (y_centers >= y_lims[0]) & (y_centers <= y_lims[1])
        
        if np.sum(row_mask) < 5: return np.empty((0,2)), np.empty((0,2))
        
        cols = np.where(col_mask)[0]
        y_active = y_centers[row_mask]
        y_buf = (y_lims[1] - y_lims[0]) * 0.005
        
        gs, es = [], []
        for c in cols:
            trace = plot_z[row_mask, c]
            peaks, props = find_peaks(trace, height=h, distance=d, prominence=p)
            if len(peaks) > 0:
                # Primary Peak (GS)
                gs_idx = peaks[np.argmax(props['peak_heights'])]
                gs_E = y_active[gs_idx]
                gs.append([x_centers[c], gs_E])
                
                # Secondary Peak (ES) - Strictly below GS
                cands = [k for k in peaks if y_active[k] < (gs_E - y_buf)]
                if cands:
                    es_idx = cands[np.argmax([y_active[k] for k in cands])]
                    es.append([x_centers[c], y_active[es_idx]])
        return np.array(gs), np.array(es)

    # Extract raw points from both windows
    Lgs, Les = extract_pts(l_roi)
    Rgs, Res = extract_pts(r_roi)
    
    # 3. Mode Selection & Fitting
    x_plot = np.linspace(x_lims[0], x_lims[1], 100)
    fit_L, fit_R = None, None
    valid_res = False
    
    # Clear all lines first
    for k in lines: lines[k].set_data([], [])

    if regime == 'Weak e-':
        # Left ES parallel to Left GS
        fit_L = constrained_fit(Lgs, Les)
        fit_R = np.polyfit(Rgs[:,0], Rgs[:,1], 1, cov=True) if len(Rgs)>2 else None
        
        if fit_L and fit_R:
            (mL, cLgs, cLes), _ = fit_L
            (mR, cRgs), _       = fit_R
            
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['L_es'].set_data(x_plot, mL*x_plot + cLes)
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            
            # Intersect: Left ES with Right GS
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRgs - cLes) / (mL - mR); y_soc = mL * x_soc + cLes
            valid_res = True

    elif regime == 'Weak h+':
        # Right ES parallel to Right GS
        fit_L = np.polyfit(Lgs[:,0], Lgs[:,1], 1, cov=True) if len(Lgs)>2 else None
        fit_R = constrained_fit(Rgs, Res)
        
        if fit_L and fit_R:
            (mL, cLgs), _       = fit_L
            (mR, cRgs, cRes), _ = fit_R
            
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            lines['R_es'].set_data(x_plot, mR*x_plot + cRes)
            
            # Intersect: Right ES with Left GS
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRes - cLgs) / (mL - mR); y_soc = mR * x_soc + cRes
            valid_res = True

    elif regime == 'Strong e-':
        # ES found in RIGHT window (Res), but parallel to LEFT GS (Lgs)
        fit_L = constrained_fit(Lgs, Res) # Res is used as "ES" here
        fit_R = np.polyfit(Rgs[:,0], Rgs[:,1], 1, cov=True) if len(Rgs)>2 else None
        
        if fit_L and fit_R:
            (mL, cLgs, cLes), _ = fit_L
            (mR, cRgs), _       = fit_R
            
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['R_es'].set_data(x_plot, mL*x_plot + cLes) # Plot as Right ES but slope L
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            
            # Intersect: "Right" ES (with Left Slope) with Right GS
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRgs - cLes) / (mL - mR); y_soc = mL * x_soc + cLes
            valid_res = True

    elif regime == 'Strong h+':
        # ES found in LEFT window (Les), but parallel to RIGHT GS (Rgs)
        fit_L = np.polyfit(Lgs[:,0], Lgs[:,1], 1, cov=True) if len(Lgs)>2 else None
        fit_R = constrained_fit(Rgs, Les) # Les is used as "ES" here
        
        if fit_L and fit_R:
            (mL, cLgs), _       = fit_L
            (mR, cRgs, cRes), _ = fit_R
            
            lines['L_gs'].set_data(x_plot, mL*x_plot + cLgs)
            lines['L_es'].set_data(x_plot, mR*x_plot + cRes) # Plot as Left ES but slope R
            lines['R_gs'].set_data(x_plot, mR*x_plot + cRgs)
            
            # Intersect: "Left" ES (with Right Slope) with Left GS
            x_ref = (cRgs - cLgs) / (mL - mR); y_ref = mL * x_ref + cLgs
            x_soc = (cRes - cLgs) / (mL - mR); y_soc = mR * x_soc + cRes
            valid_res = True

    # 4. Final Updates
    current_fits['L'] = fit_L; current_fits['R'] = fit_R
    
    # Points
    points['L_gs'].set_data(Lgs[:,0], Lgs[:,1]) if len(Lgs) else points['L_gs'].set_data([],[])
    points['L_es'].set_data(Les[:,0], Les[:,1]) if len(Les) else points['L_es'].set_data([],[])
    points['R_gs'].set_data(Rgs[:,0], Rgs[:,1]) if len(Rgs) else points['R_gs'].set_data([],[])
    points['R_es'].set_data(Res[:,0], Res[:,1]) if len(Res) else points['R_es'].set_data([],[])
    
    if valid_res:
        soc_val = np.abs(y_ref - y_soc) * 1000
        markers['ref'].set_data([x_ref], [y_ref])
        markers['soc'].set_data([x_soc], [y_soc])
        arrow_ann.set_visible(True); arrow_ann.xy = (x_soc, y_ref); arrow_ann.set_position((x_soc, y_soc))
        text_ann.set_text(f"{soc_val:.1f} µeV")
        text_ann.set_position((x_soc + (x_lims[1]-x_lims[0])*0.02, (y_ref+y_soc)/2))
        ax.set_title(f"[{name} | {regime}] $\\Delta E_{{SOC}}$=: {soc_val:.2f} µeV", fontsize=12)
    else:
        markers['ref'].set_data([],[]); markers['soc'].set_data([],[])
        arrow_ann.set_visible(False); text_ann.set_text(""); ax.set_title(f"[{name}] Insufficient Data")

    fig.canvas.draw_idle()

# --- 5. MONTE CARLO ---
def run_mc(event):
    if not current_fits['L'] or not current_fits['R']: return
    ax.set_title("Running MC Simulation..."); fig.canvas.draw()
    
    regime = radio.value_selected
    N = 5000
    
    # Sample Left
    if len(current_fits['L'][0]) == 3: # Constrained Left (m, c1, c2)
        p, cov = current_fits['L']
        sL = np.random.multivariate_normal(p, cov, N)
        mL, cL1, cL2 = sL[:,0], sL[:,1], sL[:,2]
    else: # Simple Left (m, c)
        p, cov = current_fits['L']
        sL = np.random.multivariate_normal(p, cov, N)
        mL, cL1, cL2 = sL[:,0], sL[:,1], None

    # Sample Right
    if len(current_fits['R'][0]) == 3: # Constrained Right (m, c1, c2)
        p, cov = current_fits['R']
        sR = np.random.multivariate_normal(p, cov, N)
        mR, cR1, cR2 = sR[:,0], sR[:,1], sR[:,2]
    else: # Simple Right (m, c)
        p, cov = current_fits['R']
        sR = np.random.multivariate_normal(p, cov, N)
        mR, cR1, cR2 = sR[:,0], sR[:,1], None

    # Logic per regime
    if regime == 'Weak e-':
        x_r = (cR1 - cL1)/(mL - mR); y_r = mL*x_r + cL1
        x_s = (cR1 - cL2)/(mL - mR); y_s = mL*x_s + cL2
    elif regime == 'Weak h+':
        x_r = (cR1 - cL1)/(mL - mR); y_r = mL*x_r + cL1
        x_s = (cR2 - cL1)/(mL - mR); y_s = mR*x_s + cR2
    elif regime == 'Strong e-':
        # ES is cL2 (from Left Constrained fit) intersecting Right GS (cR1)
        x_r = (cR1 - cL1)/(mL - mR); y_r = mL*x_r + cL1
        x_s = (cR1 - cL2)/(mL - mR); y_s = mL*x_s + cL2
    elif regime == 'Strong h+':
        # ES is cR2 (from Right Constrained fit) intersecting Left GS (cL1)
        x_r = (cR1 - cL1)/(mL - mR); y_r = mL*x_r + cL1
        x_s = (cR2 - cL1)/(mL - mR); y_s = mR*x_s + cR2

    diffs = np.abs(y_r - y_s) * 1000
    update_all(None)
    ax.set_title(f"[{name} | {regime}] MC: $\\Delta E_{{SOC}}$={np.mean(diffs):.2f} ± {np.std(diffs):.2f} µeV", color='green')
    print(f"[{name} | {regime}] MC Result: E_SOC = {np.mean(diffs):.4f} +/- {np.std(diffs):.4f} µV")

def recalc_data():
    """Applies filters to z_original, respecting the CURRENT ZOOM for centering."""
    global plot_z
    
    # 1. Start with raw data
    temp_z = z_original.copy()
    
    # 2. Apply Centering (Targeting the VISIBLE region)
    if use_center or use_median:
        x_min, x_max = widgets['x_min'].val, widgets['x_max'].val
        y_min, y_max = widgets['y_min'].val, widgets['y_max'].val
        
        c_mask = (x_centers >= x_min) & (x_centers <= x_max)
        r_mask = (y_centers >= y_min) & (y_centers <= y_max)
        
        # Determine offset based on visible data
        if np.any(c_mask) and np.any(r_mask):
            view_subset = temp_z[np.ix_(r_mask, c_mask)]
            
            if use_center:
                # Geometric Midpoint ((Min+Max)/2)
                offset = (np.min(view_subset) + np.max(view_subset)) / 2
            else: 
                # Median (Most common value)
                offset = np.median(view_subset)
        else:
            # Fallback for empty view
            offset = np.median(temp_z) if use_median else (np.min(temp_z)+np.max(temp_z))/2
            
        temp_z = temp_z - offset
        
    # 3. Apply Absolute Value
    if use_abs:
        temp_z = np.abs(temp_z)
        cbar.set_label(r"$\left| I_\text{AC,x} \right|$ (pA)")
    else:
        cbar.set_label(r"$I_\text{AC,x}$ (pA)")
        
    # 4. Update Global State
    plot_z = temp_z
    
    # 5. Refresh Plot
    pcm.set_array(plot_z.ravel())
    
    # Trigger the standard update (for line fits and colorbar)
    update_all(None)

def toggle_abs(event):
    global use_abs
    use_abs = not use_abs
    btn_abs.color = 'orange' if use_abs else 'lightgray'
    btn_abs.label.set_text('Abs(Z) ON' if use_abs else 'Toggle Abs(Z)')
    recalc_data()

def toggle_center(event):
    global use_center, use_median
    
    # Toggle self
    use_center = not use_center
    
    # If turning ON, force Median OFF
    if use_center:
        use_median = False
        btn_median.color = 'lightgray'
        btn_median.label.set_text('Toggle Median(Z)')
        
    btn_center.color = 'orange' if use_center else 'lightgray'
    btn_center.label.set_text('Center(Z) ON' if use_center else 'Toggle Center(Z)')
    recalc_data()

def toggle_median(event):
    global use_center, use_median
    
    # Toggle self
    use_median = not use_median
    
    # If turning ON, force Center OFF
    if use_median:
        use_center = False
        btn_center.color = 'lightgray'
        btn_center.label.set_text('Toggle Center(Z)')
        
    btn_median.color = 'orange' if use_median else 'lightgray'
    btn_median.label.set_text('Median ON' if use_median else 'Toggle Median(Z)')
    recalc_data()

# --- 6. PRINT CURRENT PARAMETERS ON CLOSE ---
def on_close(event):
    print("\n" + "="*40 + "\n      SAVED PARAMETERS\n" + "="*40)
    print(f"ax.set_xlim([{widgets['x_min'].val:.4f}, {widgets['x_max'].val:.4f}])")
    print(f"ax.set_ylim([{widgets['y_min'].val:.4f}, {widgets['y_max'].val:.4f}])")
    print(f"roi_L = ({widgets['L_min'].val:.4f}, {widgets['L_max'].val:.4f})")
    print(f"roi_R = ({widgets['R_min'].val:.4f}, {widgets['R_max'].val:.4f})")
    print(f"peaks = {{'h': {widgets['height'].val:.4f}, 'd': {int(widgets['dist'].val)}, 'p': {widgets['prom'].val:.4f}}}")
    print("="*40 + "\n")

# Connect Callbacks
for w in widgets.values(): w.on_changed(update_all)
btn_mc.on_clicked(run_mc)
btn_abs.on_clicked(toggle_abs)
btn_center.on_clicked(toggle_center)
btn_median.on_clicked(toggle_median)
radio.on_clicked(update_all)
fig.canvas.mpl_connect('close_event', on_close)

# Start
update_all(None)
plt.show(block=True)

[C1108 1h high res diamond | Strong h+] MC Result: E_SOC = 2931.9256 +/- 95.1113 µV
